# Hotel Booking Demand — Cancellation Risk & Lead Time

End-to-end ML case study using the Hotel Booking Demand dataset. The cancellation model retains `lead_time` because it is known when a booking is made; post-outcome fields are excluded to prevent leakage.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier, GradientBoostingRegressor
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error, r2_score


In [ ]:
data = pd.read_csv('hotel_bookings.csv')
df = data.drop_duplicates().copy()
df['children'] = df['children'].fillna(0)
df['country'] = df['country'].fillna(df['country'].mode()[0])
df = df.drop(columns=['company','agent'])
df['total_guests'] = df['adults'] + df['children'] + df['babies']
df['total_nights'] = df['stays_in_weekend_nights'] + df['stays_in_week_nights']
df['revenue_proxy'] = df['adr'] * df['total_nights']
print('Raw rows:', len(data), '| After deduplication:', len(df))

## Exploratory analysis

Lead time is strongly associated with cancellation behaviour; cancelled bookings are typically made further in advance. Market segment and seasonality also show meaningful differences.

In [ ]:
cancel_rate = df.groupby('market_segment')['is_canceled'].mean().sort_values(ascending=False)
cancel_rate.plot(kind='bar', figsize=(10,5), title='Cancellation Rate by Market Segment')
plt.ylabel('Cancellation Rate'); plt.tight_layout(); plt.show()
print('lead_time correlation:', round(df.select_dtypes('number').corr(numeric_only=True)['is_canceled']['lead_time'],2))

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(data=df, x='is_canceled', y='lead_time')
plt.title('Lead Time Distribution by Cancellation Status')
plt.xlabel('Cancellation Status (0 = Kept, 1 = Cancelled)'); plt.ylabel('Lead Time (days)'); plt.tight_layout(); plt.show()

## Cancellation classification

Target: `is_canceled`. `lead_time` is retained because it is available at booking time. `reservation_status` and `reservation_status_date` are excluded because they are only known after the outcome is resolved.

In [ ]:
y = df['is_canceled']
leak_cols = ['reservation_status','reservation_status_date']
X = df.drop(columns=['is_canceled'] + leak_cols)
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=.2,random_state=42,stratify=y)
num_cols = X_train.select_dtypes(include='number').columns.tolist()
cat_cols = X_train.select_dtypes(exclude='number').columns.tolist()
prep = ColumnTransformer([('num',Pipeline([('imp',SimpleImputer(strategy='median')),('scale',StandardScaler())]),num_cols),('cat',Pipeline([('imp',SimpleImputer(strategy='most_frequent')),('ohe',OneHotEncoder(handle_unknown='ignore',sparse_output=False))]),cat_cols)])
models = {'Logistic Regression':LogisticRegression(max_iter=2000,class_weight='balanced',random_state=42),'Random Forest':RandomForestClassifier(n_estimators=200,max_depth=15,class_weight='balanced_subsample',random_state=42,n_jobs=-1)}
results=[]; fitted={}
for name,model in models.items():
    pipe=Pipeline([('prep',prep),('model',model)]); pipe.fit(X_train,y_train); pred=pipe.predict(X_test); fitted[name]=pipe
    results.append([name,accuracy_score(y_test,pred),f1_score(y_test,pred)])
results_clf=pd.DataFrame(results,columns=['Model','Accuracy','F1']).round(4); print(results_clf)

In [ ]:
rf=fitted['Random Forest']
Xt=rf.named_steps['prep'].transform(X_test.iloc[:500])
feature_names=np.array(num_cols+list(rf.named_steps['prep'].named_transformers_['cat']['ohe'].get_feature_names_out(cat_cols)))
sv=shap.TreeExplainer(rf.named_steps['model']).shap_values(Xt)
sv=sv[1] if isinstance(sv,list) else (sv[:,:,1] if getattr(sv,'ndim',0)==3 else sv)
shap.summary_plot(sv,Xt,feature_names=feature_names,max_display=15,show=False)
plt.title('SHAP Summary — Top Drivers of Cancellation Risk'); plt.tight_layout(); plt.savefig('shap_classification.png',dpi=150,bbox_inches='tight'); plt.show()

## Lead-time regression

Target: `lead_time`. It is excluded from the regression feature set because it is the value being predicted.

In [ ]:
yr=df['lead_time']; Xr=df.drop(columns=['lead_time','is_canceled']+leak_cols)
Xtr,Xte,ytr,yte=train_test_split(Xr,yr,test_size=.2,random_state=42)
rn=Xtr.select_dtypes(include='number').columns.tolist(); rc=Xtr.select_dtypes(exclude='number').columns.tolist()
rprep=ColumnTransformer([('num',Pipeline([('imp',SimpleImputer(strategy='median')),('scale',StandardScaler())]),rn),('cat',Pipeline([('imp',SimpleImputer(strategy='most_frequent')),('ohe',OneHotEncoder(handle_unknown='ignore',sparse_output=False))]),rc)])
reg={'Ridge Regression':Ridge(alpha=1.0),'Gradient Boosting':GradientBoostingRegressor(n_estimators=200,max_depth=5,learning_rate=.1,random_state=42)}
for name,model in reg.items():
    pipe=Pipeline([('prep',rprep),('model',model)]); pipe.fit(Xtr,ytr); pred=pipe.predict(Xte)
    print(name,'MAE:',round(mean_absolute_error(yte,pred),2),'R2:',round(r2_score(yte,pred),4))

In [ ]:
gb=Pipeline([('prep',rprep),('model',GradientBoostingRegressor(n_estimators=200,max_depth=5,learning_rate=.1,random_state=42))]); gb.fit(Xtr,ytr); pred=gb.predict(Xte)
res=yte-pred; plt.scatter(pred,res,s=5,alpha=.25); plt.axhline(0,ls='--'); plt.xlabel('Predicted Lead Time (days)'); plt.ylabel('Residual (Actual - Predicted)'); plt.title('Residual Plot — Gradient Boosting'); plt.tight_layout(); plt.savefig('regression_residuals.png',dpi=150); plt.show()

In [ ]:
Xrt=gb.named_steps['prep'].transform(Xte.iloc[:500])
rnames=np.array(rn+list(gb.named_steps['prep'].named_transformers_['cat']['ohe'].get_feature_names_out(rc)))
rsv=shap.TreeExplainer(gb.named_steps['model']).shap_values(Xrt)
shap.summary_plot(rsv,Xrt,feature_names=rnames,max_display=15,show=False); plt.title('SHAP Summary — Top Drivers of Lead Time Prediction'); plt.tight_layout(); plt.savefig('shap_regression.png',dpi=150,bbox_inches='tight'); plt.show()